# HDGPSO demo: XGBoost and MLPClassifier with confusion matrices

This notebook runs two short end-to-end examples using HDGPSO to tune
hyperparameters, then trains the final model with the best parameters
and evaluates the predictions on a held-out test set.

The first example is a binary-classification task on the
Wisconsin Diagnostic Breast Cancer dataset, with XGBoost as the model.
The second example is a 10-class classification task on the scikit-learn
digits dataset, with an MLP as the model.


## 1. Setup

First, install the required packages. The cell below installs
`hdgpso` directly from GitHub together with the other dependencies
used in this notebook. If the packages are already installed, pip
will report that and move on quickly. Restart the kernel after
installation if you ran this in a fresh environment.


In [ ]:
# Install hdgpso and the demo dependencies.
# Comment out any line that you already have installed.
!pip install --quiet git+https://github.com/ashuxen/hdgpso.git
!pip install --quiet xgboost matplotlib scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import hdgpso
from hdgpso import HDGPSO, SearchSpace, Float, Int, Categorical
print(f"hdgpso version: {hdgpso.__version__}")

## 2. Example 1 — XGBoost on Breast Cancer

We tune six XGBoost hyperparameters using HDGPSO with the validation
loss defined as one minus the mean cross-validated accuracy. After the
search finishes, we refit XGBoost on the full training set using the
best parameters and report accuracy and a confusion matrix on a
held-out test split.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)
print(f"Train shape: {X_train_bc.shape}, Test shape: {X_test_bc.shape}")
print(f"Class balance (train): {np.bincount(y_train_bc).tolist()}")

In [ ]:
from xgboost import XGBClassifier

space_xgb = SearchSpace({
    "n_estimators":     Int(50, 400),
    "max_depth":        Int(2, 10),
    "learning_rate":    Float(1e-3, 0.3, log=True),
    "subsample":        Float(0.5, 1.0),
    "colsample_bytree": Float(0.5, 1.0),
    "min_child_weight": Int(1, 10),
})


def objective_xgb(params):
    model = XGBClassifier(
        **params,
        random_state=0,
        n_jobs=1,
        eval_metric="logloss",
        tree_method="hist",
        verbosity=0,
    )
    score = cross_val_score(
        model, X_train_bc, y_train_bc, cv=3, scoring="accuracy", n_jobs=1
    ).mean()
    return -score   # lower is better

print("Search space dimensions:", space_xgb.n_dims)

In [ ]:
result_xgb = HDGPSO(
    space_xgb,
    objective_xgb,
    population_size=8,
    iterations=8,
    seed=0,
    verbose=True,
).optimize()

print()
print("Best CV accuracy :", -result_xgb.best_loss)
print("Best parameters  :")
for k, v in result_xgb.best_params.items():
    print(f"  {k:<18} {v}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# Train the final model with the best parameters on the full training set
final_xgb = XGBClassifier(
    **result_xgb.best_params,
    random_state=0,
    n_jobs=1,
    eval_metric="logloss",
    tree_method="hist",
    verbosity=0,
)
final_xgb.fit(X_train_bc, y_train_bc)
y_pred_bc = final_xgb.predict(X_test_bc)

acc = accuracy_score(y_test_bc, y_pred_bc)
print(f"Test accuracy: {acc:.4f}")
print()
print(classification_report(
    y_test_bc, y_pred_bc, target_names=["malignant", "benign"]
))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_bc, y_pred_bc),
    display_labels=["malignant", "benign"],
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("XGBoost on Breast Cancer (test set)")
plt.tight_layout()
plt.show()

## 3. Example 2 — MLP on Digits (10-class)

The second example uses a small multi-layer perceptron on the
scikit-learn digits dataset. The search space here covers the number
of hidden units in each of two layers, the activation function, the
solver, the L2 regularization strength `alpha`, the initial learning
rate, and the maximum number of training iterations.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

X_dg, y_dg = load_digits(return_X_y=True)
scaler = StandardScaler().fit(X_dg)
X_dg = scaler.transform(X_dg)

X_train_dg, X_test_dg, y_train_dg, y_test_dg = train_test_split(
    X_dg, y_dg, test_size=0.2, random_state=42, stratify=y_dg
)
print(f"Train shape: {X_train_dg.shape}, Test shape: {X_test_dg.shape}")
print(f"Class count: {len(np.unique(y_dg))}")

In [ ]:
space_mlp = SearchSpace({
    "hidden_1":      Int(16, 128),
    "hidden_2":      Int(16, 128),
    "activation":    Categorical(["relu", "tanh"]),
    "solver":        Categorical(["adam", "lbfgs"]),
    "alpha":         Float(1e-6, 1e-2, log=True),
    "learning_rate_init": Float(1e-4, 1e-1, log=True),
    "max_iter":      Int(100, 400),
})


def objective_mlp(params):
    p = dict(params)
    h1, h2 = p.pop("hidden_1"), p.pop("hidden_2")
    p["hidden_layer_sizes"] = (h1, h2)
    if p["solver"] == "lbfgs":
        # learning_rate_init is unused for lbfgs; drop to avoid sklearn warnings
        p.pop("learning_rate_init", None)
    model = MLPClassifier(**p, random_state=0)
    score = cross_val_score(
        model, X_train_dg, y_train_dg, cv=3, scoring="accuracy", n_jobs=1
    ).mean()
    return -score

print("Search space dimensions:", space_mlp.n_dims)

In [ ]:
result_mlp = HDGPSO(
    space_mlp,
    objective_mlp,
    population_size=8,
    iterations=8,
    seed=0,
    verbose=True,
).optimize()

print()
print("Best CV accuracy :", -result_mlp.best_loss)
print("Best parameters  :")
for k, v in result_mlp.best_params.items():
    print(f"  {k:<20} {v}")

In [ ]:
# Train the final MLP with the best parameters
p = dict(result_mlp.best_params)
h1, h2 = p.pop("hidden_1"), p.pop("hidden_2")
p["hidden_layer_sizes"] = (h1, h2)
if p["solver"] == "lbfgs":
    p.pop("learning_rate_init", None)

final_mlp = MLPClassifier(**p, random_state=0)
final_mlp.fit(X_train_dg, y_train_dg)
y_pred_dg = final_mlp.predict(X_test_dg)

acc = accuracy_score(y_test_dg, y_pred_dg)
print(f"Test accuracy: {acc:.4f}")
print()
print(classification_report(y_test_dg, y_pred_dg))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_dg, y_pred_dg),
    display_labels=list(range(10)),
).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("MLP on Digits (test set)")
plt.tight_layout()
plt.show()

## 4. Notes

Both examples use a small population (8) and a small number of
iterations (8) to keep the runtime short. For real applications you
would typically use a larger budget. The defaults of `HDGPSO`
correspond to the paper settings: `F = 0.8`, `CR = 0.5`,
`c1 = c2 = 2.0`, inertia decaying from 0.7 to 0.4, and the surrogate
filter refitting every 4 iterations once at least 12 history points
are available.

The history of every trial is returned in `result.history`. This is a
pandas DataFrame and is useful for plotting convergence curves or
exporting to CSV.


In [ ]:
print("XGBoost trial history (first 5 rows):")
print(result_xgb.history.head())
print()
print(f"Total XGBoost evaluations: {result_xgb.n_evals}")
print(f"Total MLP evaluations:     {result_mlp.n_evals}")